[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/56_qk_norm_attention_solution.ipynb)

# 🟡 Solution: QK Norm Attention

Reference solution for `qk_norm_attention`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math


In [ ]:
# ✅ SOLUTION

def qk_norm_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                      mask: torch.Tensor | None = None, eps: float = 1e-6,
                      scale: float | None = None) -> torch.Tensor:
    qn = q / q.norm(dim=-1, keepdim=True).clamp_min(eps)
    kn = k / k.norm(dim=-1, keepdim=True).clamp_min(eps)
    if scale is None:
        scale = math.sqrt(q.shape[-1])
    scores = (qn @ kn.transpose(-2, -1)) * scale
    if mask is not None:
        while mask.ndim < scores.ndim:
            mask = mask.unsqueeze(-2)
        scores = scores.masked_fill(~mask, float('-inf'))
    probs = torch.softmax(scores, dim=-1)
    return probs @ v


In [ ]:
# Verify
q = torch.randn(2, 4, 5, 8)
k = torch.randn(2, 4, 6, 8)
v = torch.randn(2, 4, 6, 8)
print(qk_norm_attention(q, k, v).shape)


In [ ]:
# Run judge
from torch_judge import check
check('qk_norm_attention')
